## 2.1.3 工作流程图解


In [19]:

import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from pywin.debugger.fail import e
from rich import print as rprint

# ---------------------------------------------------------------------------
# 1. 环境变量与模型初始化
# ---------------------------------------------------------------------------

# 从 .env 文件中加载环境变量；override=True 表示同名变量以 .env 文件为准
load_dotenv(override=True)

# 从环境变量中读取 DeepSeek 的 API Key 和 API Base 地址
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_BASE = os.getenv("DEEPSEEK_API_BASE")

# 初始化聊天模型：
# model 采用 "provider:model_name" 的格式指定 DeepSeek 的模型
# api_key / base_url 用于覆盖默认配置，指向自定义的 DeepSeek 服务
model = init_chat_model(
      model="deepseek:deepseek-v4-flash",
      api_key=DEEPSEEK_API_KEY,
      base_url=DEEPSEEK_API_BASE,
      extra_body={"thinking": {"type": "disabled"}}  # 关键：永久关闭思考模式
)

### 第 1 步：定义结构

In [3]:
from pydantic import BaseModel, Field


class BookInfo(BaseModel):
      title: str = Field(description=" 书名 ")
      author: str = Field(description=" 作者名字 ")
      tags: list[str] = Field(description=" 书籍的标签或分类 ")

## 2.2 模式 2 ： TypedDict
TypedDict 是 Python 3.8+ 引入的一种类型提示工具， 即带有类型声明的字典结构 。适合需要快速定义
字典结构且无需 Pydantic 重量级功能的场景。

In [ ]:
# 普通 dict 没有类型信息：

{
      "title": " 盗梦空间 ",
      "year": 2010,
      "director": " 克里斯托弗 · 诺兰 ",
      "rating": 9.3
}


In [4]:
from typing_extensions import TypedDict


class MovieDict(TypedDict):
      title: str
      year: int
      director: str
      rating: float


movie: MovieDict = {
      "title1": " 盗梦空间 ",
      "year": 2010,
      "director": " 克里斯托弗 · 诺兰 ",
      "rating": 8.8,
}

print(movie)

{'title1': ' 盗梦空间 ', 'year': 2010, 'director': ' 克里斯托弗 · 诺兰 ', 'rating': 8.8}


 ### 2.2.2 基本使用
Annotated 的使用 替代 Field 来写说明书配合 LangChain 用得最多

In [6]:
"""
使用 TypedDict 模型定义结构化输出
"""

from typing_extensions import TypedDict, Annotated


class MovieTypedDict(TypedDict):
      """
      电影的详细信息
      """

      title: Annotated[str, " 电影的正式名称，例如《盗梦空间》 "]
      year: Annotated[int, " 电影的公映年份，使用四位数字表示 "]
      director: Annotated[str, " 电影导演的全名 "]
      rating: Annotated[float, " 电影在 10 分制下的评分，可包含一位小数 "]


# 设置模型结构化输出
structured_llm = model.with_structured_output(MovieTypedDict)

# 调用模型并获取结构化输出
response = structured_llm.invoke(" 给我介绍下电影《星际穿越》 ")

print(type(response))
print(response)

<class 'dict'>
{'title': '星际穿越', 'year': 2014, 'director': '克里斯托弗·诺兰', 'rating': 9.4}


**举例 2：** 返回嵌套结构

In [8]:
from typing import TypedDict, List, Annotated


# 使用 TypedDict 定义嵌套结构
class Actor(TypedDict):
      """演员情况"""

      name: Annotated[str, " 演员姓名 "]
      role: Annotated[str, " 饰演的角色 "]


class Movie(TypedDict):
      """电影情况"""

      # title: Annotated[str, " 电影标题 "]
      title: Annotated[str, "电影标题"]
      year: Annotated[int, " 上映年份 "]
      director: Annotated[str, " 导演 "]
      cast: Annotated[List[Actor], " 演员列表 "]  # 嵌套列表定义
      rating: Annotated[float, " 评分 "]


# 设置模型结构化输出
structured_llm = model.with_structured_output(Movie)

# 调用模型并获取结构化输出
resp = structured_llm.invoke(" 给我介绍下电影《盗梦空间》 ")

# 访问嵌套数据
print(f" 电影名 : {resp['title']}")
print(f" 上映年份 : {resp['year']}")
print(f" 导演 : {resp['director']}")
print(f" 演员列表 :{resp['cast']}")
print(f" 评分 : {resp['rating']}")

 电影名 : 盗梦空间
 上映年份 : 2010
 导演 : 克里斯托弗·诺兰
 演员列表 :[{'name': '莱昂纳多·迪卡普里奥', 'role': '多姆·柯布'}, {'name': '约瑟夫·高登-莱维特', 'role': '亚瑟'}, {'name': '艾利奥特·佩吉', 'role': '阿丽雅德妮'}, {'name': '汤姆·哈迪', 'role': '伊姆斯'}, {'name': '渡边谦', 'role': '齐藤'}, {'name': '玛丽昂·歌迪亚', 'role': '梅尔'}, {'name': '基里安·墨菲', 'role': '罗伯特·费舍尔'}, {'name': '迈克尔·凯恩', 'role': '迈尔斯'}]
 评分 : 9.4


In [15]:
from typing_extensions import TypedDict, Annotated


class MovieDict(TypedDict):
      """
      电影的详细信息
      """

      title: Annotated[str, ..., " 电影标题 "]
      year: Annotated[int, ..., " 电影上映年份 "]
      director: Annotated[str, ..., " 导演 "]
      rating: Annotated[float, ..., " 电影评分，满分十分 "]


model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke(
      " 根据这段话抽取盗梦空间的信息，不包含的信息可以留空：盗梦空间在 2010 年上映，导演是克里斯托弗 · 诺兰。 "
)
print(response)
print(type(response))

{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗 · 诺兰', 'rating': 0}
<class 'dict'>


In [16]:
from typing_extensions import TypedDict, Annotated


class MovieDict(TypedDict):
      """
      电影的详细信息
      """

      title: Annotated[str, ..., " 电影标题 "]
      year: Annotated[int, ..., " 电影上映年份 "]
      director: Annotated[str, Ellipsis, " 导演 "]
      rating: Annotated[float, " 电影评分，满分十分 "]


model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke(
      " 根据这段话抽取盗梦空间的信息，不包含的信息必须留空：盗梦空间上映于 2010 年，我们并不知道它的导演是谁。 "
)
print(response)
print(type(response))

{'title': '盗梦空间', 'year': 2010, 'director': ''}
<class 'dict'>


### 2.3 模式 3 ： JSON Schema

In [20]:

json_schema = {
      "title": "Movie",
      "description": "A movie with details",
      "type": "object",
      "properties": {
            "title": {"type": "string", "description": "The title of the movie"},
            "year": {"type": "integer", "description": "The year the movie was released"},
            "director": {"type": "string", "description": "The director of the movie"},
            "rating": {"type": "number", "description": "The movie's rating out of 10"},
      },
      "required": ["title", "year", "director", "rating"],
}

structured_model = model.with_structured_output(json_schema, method="json_schema")

response = structured_model.invoke(" 给出盗梦空间的信息 ")
print(response)
print(type(response))

{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'rating': 8.8}
<class 'dict'>


### **举例 2：** 返回嵌套结构

In [24]:
"""
使用 JSON Schema 定义嵌套结构
"""

# 1. 定义嵌套的 JSON Schema
project_schema = {
      "title": "MovieInfo",
      "description": " 包含电影标题、上映年份、导演、演员和评分的电影对象 ",
      "type": "object",
      "properties": {
            "title": {"type": "string", "description": " 电影标题 "},
            "year": {"type": "integer", "description": " 上映年份 "},
            "director": {"type": "string", "description": " 导演 "},
            "cast": {  # 定义嵌套数组
                  "type": "array",
                  "description": " 演员列表 ",
                  "items": {
                        "type": "object",
                        "properties": {
                              "name": {"type": "string", "description": " 演员姓名 "},
                              "role": {"type": "string", "description": " 演员角色 "},
                        },
                        "required": ["name", "role"],
                  },
            },
            "rating": {"type": "number", "description": " 评分（ 10 分制） "},
      },
      "required": ["title", "year", "director", "cast", "rating"],
}

# 绑定 JSON Schema 到模型
structured_model = model.with_structured_output(project_schema)

# 调用模型
response = structured_model.invoke(
      " 生成一个关于《星际穿越》的电影信息，包含导演、演员、评分 "
)
rprint(response)

{
    'title': '星际穿越',
    'year': 2014,
    'director': '克里斯托弗·诺兰',
    'cast': [
        {'name': '马修·麦康纳', 'role': '库珀'},
        {'name': '安妮·海瑟薇', 'role': '布兰德博士'},
        {'name': '杰西卡·查斯坦', 'role': '墨菲（成年）'},
        {'name': '迈克尔·凯恩', 'role': '布兰德教授'},
        {'name': '麦肯基·弗依', 'role': '墨菲（幼年）'},
        {'name': '马特·达蒙', 'role': '曼恩博士'}
    ],
    'rating': 9.4
}

### 2.4 模式 4 ： @dataclass
给类加上 @dataclass 后， Python 会根据字段声明自动生成常用方法，例如：
__init__
__repr__
__eq__

In [25]:
from dataclasses import dataclass


@dataclass
class Movie:
      title: str
      year: int
      director: str
      rating: float

In [26]:
from pydantic import Field
from dataclasses import dataclass


@dataclass
class Movie:
      """
      电影的详细信息
      """

      title: str = Field(description=" 电影标题 ")
      year: int = Field(description=" 电影上映年份 ")
      director: str = Field(description=" 导演 ")
      rating: float = Field(description=" 电影评分，满分十分 ")


structured_model = model.with_structured_output(Movie)

response = structured_model.invoke(" 给出盗梦空间的信息 ")
print(response)
print(type(response))

{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'rating': 9.4}
<class 'dict'>


# 3、 关于类型校验  Pydantic
用 Pydantic 定义 schema ，在接收到响应后会进行校验，字段不匹配则抛出异常，其余三种方式不校验。

In [ ]:
from pydantic import BaseModel, Field, SecretStr

from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_base="http://localhost:8889",
    api_key=SecretStr("<KEY>"),
)


class MovieModel(BaseModel):
    """
    电影的详细信息
    """

    title: str = Field(description=" 电影标题 ")
    year: int = Field(description=" 电影上映年份 ")
    director: str = Field(description=" 导演 ")
    rating: float = Field(description=" 电影评分，满分十分 ")


model_with_structure = model.with_structured_output(MovieModel)
response = model_with_structure.invoke(" 给出盗梦空间的信息 ")
print(response)
print(type(response))

### 3.2.2 TypedDict

In [3]:
from typing_extensions import TypedDict, Annotated

from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_base="http://localhost:8889",
    api_key=SecretStr("<KEY>"),
)


class MovieDict(TypedDict):
    """
    电影的详细信息
    """

    title: Annotated[str, " 电影标题 "]
    year: Annotated[int, " 电影上映年份 "]
    director: Annotated[str, " 导演 "]
    rating: Annotated[float, " 电影评分，满分十分 "]


structured_model = model.with_structured_output(MovieDict)
response = structured_model.invoke(" 给出盗梦空间的信息 ")
print(response)
print(type(response))

{'title1': '盗梦空间', 'year2': 2010, 'director': '克里斯托弗·诺兰', 'rating': 9.3}
<class 'dict'>


### 3.2.3 JSON schema

In [4]:
from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_base="http://localhost:8889",
    api_key=SecretStr("<KEY>"),
)

json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {"type": "string", "description": "The title of the movie"},
        "year": {"type": "integer", "description": "The year the movie was released"},
        "director": {"type": "string", "description": "The director of the movie"},
        "rating": {"type": "number", "description": "The movie's rating out of 10"},
    },
    "required": ["title", "year", "director", "rating"],
}

structured_model = model.with_structured_output(json_schema, method="json_schema")

response = structured_model.invoke(" 给出盗梦空间的信息 ")
print(response)
print(type(response))

{'title1': '盗梦空间', 'year2': 2010, 'director': '克里斯托弗·诺兰', 'rating': 9.3}
<class 'dict'>


### 3.3.4 @dataclass


In [5]:
from dataclasses import dataclass

from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_base="http://localhost:8889",
    api_key=SecretStr("<KEY>"),
)


@dataclass
class Movie:
    """
    电影的详细信息
    """

    title: str = Field(description=" 电影标题 ")
    year: int = Field(description=" 电影上映年份 ")
    director: str = Field(description=" 导演 ")
    rating: float = Field(description=" 电影评分，满分十分 ")


structured_model = model.with_structured_output(Movie)

response = structured_model.invoke(" 给出盗梦空间的信息 ")
print(response)
print(type(response))

{'title1': '盗梦空间', 'year2': 2010, 'director': '克里斯托弗·诺兰', 'rating': 9.3}
<class 'dict'>
